# Handwritten Digit Classifier: end-to-end pipeline

Recognising handwritten digits (0–9) from images with a convolutional neural network.
This notebook runs every stage the deployed API uses, with the same code from `src/`:

1. Data acquisition
2. Exploring the data: features and what they tell us
3. Preprocessing
4. Model creation
5. Training
6. Model testing with the full set of metrics
7. Prediction functions
8. Retraining on newly uploaded data

**Dataset:** 1,797 real handwritten digits from the UCI / scikit-learn digits dataset, recorded at 8×8 pixels
and stored as 64×64 grayscale PNGs in `data/raw/train` (1,437) and `data/raw/test` (360).

In [ ]:
import os, sys, random, io
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from PIL import Image

from src import config
from src import preprocessing as prep
from src import model as mdl

sns.set_theme(style="whitegrid")
tf.random.set_seed(config.SEED); np.random.seed(config.SEED); random.seed(config.SEED)
print("TensorFlow", tf.__version__)

## 1. Data acquisition

`acquire_data()` uses the PNGs already in `data/raw/`. If that folder is empty it exports the same dataset
from scikit-learn (no download needed), with a stratified 80/20 train/test split. The test set is never used
for training or model selection.

In [ ]:
counts = prep.acquire_data()
pd.DataFrame(counts).assign(total=lambda d: d.sum(axis=1))

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(9, 9))
for d in range(10):
    files = sorted((prep.RAW_DIR / "train" / str(d)).iterdir())[:10]
    for k, f in enumerate(files):
        axes[d, k].imshow(Image.open(f), cmap="gray"); axes[d, k].axis("off")
plt.suptitle("Ten raw training images of each digit"); plt.tight_layout(); plt.show()

## 2. Exploring the data

Images have no columns, so we measure features from each one (after preprocessing, see section 3):

| Feature | How it is computed | Why it matters |
|---|---|---|
| Class count | images per folder | imbalance would bias the model |
| Aspect ratio | width ÷ height of the digit's bounding box | some digits are naturally narrow |
| Ink coverage | share of pixels that are part of the stroke | how much "material" each digit has |
| Top-heaviness | share of ink in the top half | where the mass of the digit sits |
| Average image | mean of all images of a digit | how consistently people write it |

In [ ]:
prep.process_dataset()        # explained in section 3; done here so features use the final format
feats = prep.dataset_features(config.TRAIN_DIR)
feats.groupby("label")[["aspect_ratio", "ink_coverage", "top_heaviness", "stroke_intensity"]].agg(["mean", "std"]).round(3)

### Feature 1: class distribution

In [ ]:
train_counts, test_counts = prep.class_counts(config.TRAIN_DIR), prep.class_counts(config.TEST_DIR)
ax = pd.DataFrame({"train": train_counts, "test": test_counts}).plot.bar(stacked=True, figsize=(8, 4), color=["#1E5B4F", "#C98C00"])
ax.set(xlabel="Digit", ylabel="Images", title="Images per digit"); plt.xticks(rotation=0); plt.show()
print(f"Largest / smallest class: {max(train_counts.values()) / min(train_counts.values()):.2f}")

**Interpretation.** Every digit has 141–146 training images, a ratio of about 1.04. The dataset is balanced,
so the model has no incentive to favour any digit, and plain accuracy is a fair headline metric. We still
report macro-averaged precision, recall and F1 so a weak digit cannot hide behind a good average.

### Feature 2: digit width (aspect ratio)

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=feats, x="label", y="aspect_ratio", color="#DCEAE5")
plt.title("Width ÷ height of each digit"); plt.xlabel("Digit"); plt.ylabel("Aspect ratio"); plt.show()

**Interpretation.** 1 is the narrowest digit by a clear margin, so width alone nearly identifies it.
Most other digits overlap heavily: their boxes cover the same range, which means a simple rule on
width would confuse them, and the model has to learn shape (loops, corners, stroke direction). That is the
reason to use a convolutional network, which learns local shape patterns, rather than a model on summary numbers.

> ✍️ Check this against your plot and mention the widest digit.

### Feature 3: ink coverage and top-heaviness

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=feats, x="label", y="ink_coverage", color="#DCEAE5", ax=ax[0])
ax[0].set(title="Share of pixels that are ink", xlabel="Digit")
sns.boxplot(data=feats, x="label", y="top_heaviness", color="#F5E6C4", ax=ax[1])
ax[1].axhline(0.5, color="grey", ls="--"); ax[1].set(title="Share of ink in the top half", xlabel="Digit")
plt.tight_layout(); plt.show()

**Interpretation.** Ink coverage is similar for every digit (roughly 43–50% of the frame), so "how much is written"
says little; 1 uses the least ink. Top-heaviness is far more telling: 9, 7 and 5 carry most of their ink above the
middle line, 6 is strongly bottom-heavy (its loop sits low), 4 and 2 lean slightly low, and 0, 1, 3 and 8 are close
to balanced. Pairs on opposite sides of the line (6 vs 9) are easy to separate, while pairs on the same side
(5 vs 9, 3 vs 8) depend on finer detail. We expect those in the confusion matrix.

> ✍️ Confirm with your plot.

### Feature 4: the average digit

In [ ]:
means = prep.mean_images(config.TRAIN_DIR, size=32)
fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for d in range(10):
    m = np.array(means[str(d)])
    axes[0, d].imshow(m, cmap="gray"); axes[0, d].set_title(str(d)); axes[0, d].axis("off")
    stack = np.stack([np.asarray(Image.open(f), dtype=np.float32) / 255 for f in (config.TRAIN_DIR / str(d)).iterdir()])
    axes[1, d].imshow(stack.std(0), cmap="magma"); axes[1, d].axis("off")
axes[0, 0].set_ylabel("mean"); axes[1, 0].set_ylabel("std")
plt.suptitle("Top: average image of each digit. Bottom: where pixels vary most between writers"); plt.show()

**Interpretation.** Sharp averages (2 and 6 in this dataset) mean people write that digit in one consistent way.
Blurry averages, such as 5 and 9, mix several writing styles. The bottom row shows *where* writers disagree:
bright areas are the pixels that vary the most and carry the most information for telling digits apart.

## 3. Preprocessing

Steps, in order:

1. **Canonicalize** every image with `prep.canonicalize()`:
   grayscale → invert if the background is light (so photos of ink on paper match the dataset's white-on-black)
   → stretch contrast → crop to the digit → pad to a square → shrink to **8×8** (the dataset's native resolution)
   → enlarge to **32×32**. The same function runs on uploads and on the image sent to `/predict`, so the model
   always sees the same format.
2. **Load** with `image_dataset_from_directory` as single-channel 32×32 images.
3. **Split** the training folder 80/20 into train and validation (fixed seed).
4. **Batch** (32), **cache**, **shuffle** the training set and **prefetch**.
5. **Augment** (training only): rotation ±5%, shift ±10%, zoom ±10%. No flips, because a mirrored 2 is not a 2.
6. **Scale** pixels from 0–255 to 0–1 (a `Rescaling` layer inside the model).

In [ ]:
raw = sorted((prep.RAW_DIR / "train" / "5").iterdir())[0]
paper = Image.new("L", (300, 300), 230)
from PIL import ImageDraw, ImageFont
ImageDraw.Draw(paper).text((95, 40), "5", fill=20, font=ImageFont.load_default(size=170))

fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, img, t in zip(ax, [Image.open(raw), prep.canonicalize(raw), paper, prep.canonicalize(paper)],
                     ["raw dataset image (64×64)", "canonical (32×32)", "dark ink on paper", "same photo, canonical"]):
    a.imshow(img, cmap="gray", vmin=0, vmax=255); a.set_title(t, fontsize=10); a.axis("off")
plt.show()

In [ ]:
train_ds, val_ds, class_names = prep.load_datasets()
test_ds = prep.load_test_dataset(class_names=class_names)
x, y = next(iter(train_ds))
print("classes:", class_names)
print("batch:", x.shape, x.dtype, "| pixel range:", float(tf.reduce_min(x)), "to", float(tf.reduce_max(x)))

In [ ]:
aug = prep.augmentation_layer()
plt.figure(figsize=(12, 2.4))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    img = x[:1] if i == 0 else aug(x[:1], training=True)
    plt.imshow(img[0, :, :, 0], cmap="gray"); plt.axis("off"); plt.title("original" if i == 0 else "augmented", fontsize=8)
plt.show()

## 4. Model creation

A small convolutional network:

```
input 32×32×1 → augmentation → rescale to 0–1
→ [conv 32 → batch-norm → conv 32 → max-pool → dropout 0.25]
→ [conv 64 → batch-norm → conv 64 → max-pool → dropout 0.25]
→ flatten → dense 128 (ReLU, L2) → dropout 0.4 → softmax over 10 digits
```

- **Convolutions** learn local stroke patterns (curves, corners, endings) wherever they appear.
- **Batch normalisation** stabilises and speeds up training.
- **Dropout, L2 and augmentation**: three kinds of regularisation, needed with only ~1,150 training images.
- **Adam** (learning rate 0.001), **early stopping** on validation loss with best weights restored, and
  **ReduceLROnPlateau** to halve the learning rate when progress stalls.

In [ ]:
model = mdl.build_model(len(class_names))
model.summary()

## 5. Training

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=mdl.get_callbacks(), verbose=2)
history = mdl.history_dict(history)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, a in (("accuracy", ax[0]), ("loss", ax[1])):
    a.plot(history[k], label="train"); a.plot(history["val_" + k], label="validation")
    a.set(title=k.title(), xlabel="Epoch"); a.legend()
plt.show()

**Reading the curves.** Validation accuracy above training accuracy is normal here: dropout and augmentation
make training harder than validation. If validation loss starts rising while training loss keeps falling,
the model is overfitting, and early stopping restores the best epoch.

## 6. Model testing

On the 360 held-out test images:

- **Accuracy**: share classified correctly.
- **Precision** per digit: when the model says "3", how often it is a 3.
- **Recall** per digit: of all real 3s, how many it finds.
- **F1**: harmonic mean of precision and recall, macro (each digit equal) and weighted.
- **Log loss**: punishes confident mistakes, so it measures how trustworthy the probabilities are.
- **ROC-AUC (one-vs-rest)**: how well each digit is ranked above the others.
- **Confusion matrix**: which digits get mixed up.

In [ ]:
metrics = mdl.evaluate_model(model, test_ds, class_names)
pd.Series({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)}, name="test set").to_frame()

In [ ]:
pd.DataFrame(metrics["per_class"]).T.round(3)

In [ ]:
cm = np.array(metrics["confusion_matrix"])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix"); plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
y_true, y_prob = mdl.predict_dataset(model, test_ds)
y_bin = label_binarize(y_true, classes=range(10))
plt.figure(figsize=(7, 6))
for i, c in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f"{c} (AUC {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves, one-vs-rest"); plt.legend(fontsize=8); plt.show()

In [ ]:
y_pred = y_prob.argmax(1)
test_files = [f for c in class_names for f in sorted((config.TEST_DIR / c).iterdir())]
wrong = np.where(y_pred != y_true)[0]
plt.figure(figsize=(2 * len(wrong), 2.4))
for k, i in enumerate(wrong):
    plt.subplot(1, len(wrong), k + 1); plt.imshow(Image.open(test_files[i]), cmap="gray"); plt.axis("off")
    plt.title(f"true {class_names[y_true[i]]}\npred {class_names[y_pred[i]]} ({y_prob[i].max():.0%})", fontsize=9)
plt.suptitle("Every test mistake"); plt.show()

**Conclusion.** ✍️ State accuracy and macro F1, the digit with the lowest recall, and the most common confusion.
Compare with section 2: do the mistakes fall on digits with blurry averages or the same top-heaviness?

In [ ]:
version = mdl.save_model(model, class_names, metrics, history, trigger="initial: notebook")
print("Saved", config.MODEL_PATH, "as version", version)

## 7. Prediction functions

`src/prediction.py` is what the API calls: it canonicalizes the image and returns the digit, the confidence and every class probability.

In [ ]:
from src.prediction import Predictor
predictor = Predictor()

def show_prediction(image, title=""):
    r = predictor.predict(image)
    fig, ax = plt.subplots(1, 3, figsize=(11, 2.8))
    ax[0].imshow(Image.open(image) if isinstance(image, (str, Path)) else image, cmap="gray"); ax[0].set_title(title or "input"); ax[0].axis("off")
    ax[1].imshow(prep.canonicalize(image), cmap="gray"); ax[1].set_title("what the model sees"); ax[1].axis("off")
    ax[2].bar(list(r["probabilities"]), list(r["probabilities"].values()), color="#1E5B4F"); ax[2].set_ylim(0, 1)
    ax[2].set_title(f"prediction: {r['label']} ({r['confidence']:.0%})"); plt.tight_layout(); plt.show()
    return r

sample = random.choice(sorted((prep.RAW_DIR / "test").glob("*/*.png")))
show_prediction(sample, f"test image, true digit {sample.parent.name}")

In [ ]:
buf = io.BytesIO(); paper.save(buf, "PNG")
show_prediction(Image.open(buf), "a printed 5 on light paper")

## 8. Retraining

How retraining works in production (`mdl.retrain`):

1. Users upload labelled digits (dashboard or `POST /upload`). Each image is canonicalized and saved to `data/uploads/<digit>/`.
2. **Trigger:** the *Retrain model* button, or automatically once `RETRAIN_THRESHOLD` (50) images are waiting.
3. Uploads move into `data/train` (20% into `data/test` so evaluation covers the new data).
4. The current model's weights are copied into a fresh network and training continues at a lower learning rate (warm start).
5. **Champion vs challenger:** old and new model are scored on the same test set; the new one goes live only if
   accuracy does not drop by more than 1%. The old file is archived in `models/archive/`.
6. Every API container notices the new model file and reloads it.

Here we simulate users uploading their own digits by drawing new ones with different fonts.
(In a real project these would be photos of handwritten digits.)

In [ ]:
for d in range(10):
    for size in (120, 150, 180):
        img = Image.new("L", (260, 260), 235)
        ImageDraw.Draw(img).text((70, 20), str(d), fill=15, font=ImageFont.load_default(size=size))
        buf = io.BytesIO(); img.save(buf, "PNG")
        prep.save_uploads([(f"{d}_{size}.png", buf.getvalue())], label=str(d))
print("Pending uploads:", prep.pending_upload_count(), "| auto-trigger fires:", mdl.should_retrain())

In [ ]:
result = mdl.retrain(trigger="manual: notebook demo", epochs=5)
result

In [ ]:
pd.DataFrame(mdl.read_json(config.META_PATH)["versions"])

## Summary

- The data is balanced; width and where the ink sits explain which digits are easy (0, 2, 6 are classified
  perfectly) and which need fine shape detail (8, 9 and 5 have the lowest recall).
- One `canonicalize()` function prepares training data, uploads and prediction inputs, so they never drift apart.
- A small CNN with augmentation, dropout, L2 and early stopping reaches about 97% test accuracy.
- Retraining is safe: a new model replaces the old one only if it scores at least as well.